# Oryza Vision — Google Colab Auto Training v3

**Tanpa Mendeley dan tanpa upload manual.** Dataset publik langsung dari Hugging Face (`Project-AgML/rice_leaf_disease_classification`), 5 kelas, audit, split, training 2 arsitektur, evaluasi, simpan ke Drive.

> Perhatian: dataset HF memiliki total 3.829 sampel untuk 6 kelas. Notebook hanya mengambil 5 kelas (sekitar 3.197 gambar sebelum audit). Foto ini belum diverifikasi khusus fase pembibitan.


## 0 — Konfigurasi & Drive

In [ ]:
#@title Pengaturan (boleh langsung jalan)
DATASET_ID = 'n8jpkpfz5p'
DATASET_VERSION = 1
EPOCHS = 12              # Tambah jika hasil belum konvergen
BATCH_SIZE = 32          # Jika GPU out-of-memory, turunkan ke 16
LEARNING_RATE = 3e-4
SEED = 42
NUM_WORKERS = 2
IMG_SIZE = 224
SAVE_TO_DRIVE = True
RUN_BOTH_MODELS = True

from pathlib import Path
import os, random, json, shutil, sys
ROOT = Path('/content/oryza_vision')
ROOT.mkdir(parents=True, exist_ok=True)
ZIP_PATH = ROOT / 'rice_leaf_disease.zip'
RAW_DIR = ROOT / 'raw'
CURATED_DIR = ROOT / 'curated'
SPLIT_DIR = ROOT / 'processed'
REPORT_DIR = ROOT / 'reports'
MODEL_DIR = ROOT / 'models'
for p in (RAW_DIR,CURATED_DIR,SPLIT_DIR,REPORT_DIR,MODEL_DIR): p.mkdir(parents=True, exist_ok=True)
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_ROOT = Path('/content/drive/MyDrive/Oryza-Vision')
    for part in ('datasets','models','results'):
        (DRIVE_ROOT / part).mkdir(parents=True, exist_ok=True)
else: DRIVE_ROOT = None
print('Folder kerja:',ROOT)


## 1 — Unduh langsung dataset Hugging Face ke Colab

Sumber: https://huggingface.co/datasets/Project-AgML/rice_leaf_disease_classification . Dataset akan diunduh otomatis ke cache Colab (sekitar 1 GB seluruh sumber) dan diekspor menjadi folder kelas. Akses internet Colab diperlukan, tanpa token untuk dataset publik.


In [ ]:
%pip -q install -U datasets huggingface_hub pillow scikit-learn pandas matplotlib seaborn tqdm


In [ ]:
from datasets import load_dataset
from PIL import Image, ImageOps
from tqdm.auto import tqdm
import hashlib, json

HF_REPO = 'Project-AgML/rice_leaf_disease_classification'
CLASS_NAMES = ['Healthy','Leaf_Blast','Brown_Spot','Bacterial_Blight','Leaf_Scald']
ALIASES = {
    'healthy_rice_leaf': 'Healthy',
    'leaf_blast': 'Leaf_Blast',
    'brown_spot': 'Brown_Spot',
    'bacterial_leaf_blight': 'Bacterial_Blight',
    'leaf_scald': 'Leaf_Scald',
}

def slug(s):
    import re
    return re.sub(r'[^a-z0-9]+','_',s.lower()).strip('_')

print('Mengunduh dataset publik dari:', HF_REPO)
ds = load_dataset(HF_REPO, split='train')
features = ds.features
print('Jumlah sumber:',len(ds),'| Features:',features)
label_names = features['label'].names
print('Kelas asli:',label_names)

# Pastikan kelas sumber persis bisa dipetakan, sehingga tidak ada label salah diam-diam
mapping = {idx: ALIASES.get(slug(name)) for idx,name in enumerate(label_names)}
missing = set(CLASS_NAMES) - {v for v in mapping.values() if v}
if missing: raise RuntimeError(f'Nama kelas dari Hugging Face berubah. Kelas tidak ditemukan: {missing}; kelas aktual: {label_names}')

# Generate ulang agar resume tidak bercampur data lama.
shutil.rmtree(RAW_DIR, ignore_errors=True)
RAW_DIR.mkdir(parents=True, exist_ok=True)
from collections import Counter
saved=Counter(); bad=[]
for idx, sample in enumerate(tqdm(ds, desc='Ekspor 5 kelas')):
    label = mapping.get(int(sample['label']))
    if label is None: continue
    try:
        im = ImageOps.exif_transpose(sample['image']).convert('RGB')
        folder = RAW_DIR / label
        folder.mkdir(parents=True, exist_ok=True)
        im.save(folder / f'{idx:06d}.png')
        saved[label]+=1
    except Exception as e:
        bad.append((idx,label,str(e)))
print('Selesai. Jumlah per kelas:',dict(saved),'| Gambar gagal diekspor:',len(bad))
if bad: print('Contoh kesalahan:',bad[:5])
if set(saved) != set(CLASS_NAMES): raise RuntimeError('Beberapa kelas kosong. Periksa kelas asli.')


## 2 — Verifikasi folder dan label kelas

Folder dataset sudah dibuat otomatis di `/content/oryza_vision/raw` oleh cell sebelumnya.


In [ ]:
from collections import Counter
EXTS={'.jpg','.jpeg','.png','.webp','.bmp','.tif','.tiff'}
def find_label(path):
    name=path.parent.name
    return name if name in CLASS_NAMES else None
images = [p for p in RAW_DIR.rglob('*') if p.is_file() and p.suffix.lower() in EXTS]
found = Counter(find_label(p) or 'UNMAPPED' for p in images)
print('Jumlah gambar:',dict(found))
missing=[c for c in CLASS_NAMES if found[c]==0]
if missing: raise RuntimeError('Folder kelas kosong: '+str(missing))


## 3 — Audit, kurasi, deduplikasi SHA-256 dan konversi RGB
Duplikat **byte-identik** maupun konten gambar identik setelah normalisasi orientasi dibuang; **near-duplicate** (crop/edit/augmentasi) belum dideteksi otomatis. Simpan log audit untuk pengecekan manual. Foto dengan label salah tidak dapat diketahui otomatis.

In [ ]:
from PIL import Image, ImageOps, UnidentifiedImageError
import hashlib, io, pandas as pd
from tqdm.auto import tqdm

# Dibuat ulang setiap eksekusi agar hasil konsisten
shutil.rmtree(CURATED_DIR, ignore_errors=True)
CURATED_DIR.mkdir(parents=True)
rows=[]; rejected=[]; seen_pixels={}
for p in tqdm(sorted(images),desc='Audit dan kurasi'):
    label=find_label(p)
    if label is None: continue
    try:
        with Image.open(p) as im:
            im=ImageOps.exif_transpose(im).convert('RGB')
            if min(im.size)<80: raise ValueError('Resolusi terlalu kecil')
            # fingerprint pixel; mendeteksi file berbeda dengan pixel yang persis sama
            fingerprint=hashlib.sha256(str(im.size).encode()+im.tobytes()).hexdigest()
            if fingerprint in seen_pixels:
                reason='duplicate_same_label' if seen_pixels[fingerprint][0]==label else 'duplicate_label_conflict'
                rejected.append({'file':str(p.relative_to(RAW_DIR)),'label':label,'reason':reason,'duplicate_of':seen_pixels[fingerprint][1]})
                # jika lintas kelas, tandai gambar sebelumnya untuk dibuang juga
                if reason=='duplicate_label_conflict':
                    seen_pixels[fingerprint][2]=True
                continue
            short=hashlib.sha256(str(p.relative_to(RAW_DIR)).encode()).hexdigest()[:12]
            out=CURATED_DIR/label/(short+'.jpg')
            out.parent.mkdir(parents=True,exist_ok=True)
            im.save(out,'JPEG',quality=95)
            seen_pixels[fingerprint]=[label,str(p.relative_to(RAW_DIR)),False]
            rows.append({'path':str(out),'label':label,'fingerprint':fingerprint,'source':str(p.relative_to(RAW_DIR))})
    except Exception as e:
        rejected.append({'file':str(p.relative_to(RAW_DIR)),'label':label,'reason':'invalid:'+str(e)})
# Keluarkan seluruh grup duplikat lintas label, bukan hanya kemunculan belakangan
conflicts={k for k,v in seen_pixels.items() if v[2]}
kept=[]
for row in rows:
    if row['fingerprint'] in conflicts:
        Path(row['path']).unlink(missing_ok=True)
        rejected.append({'file':row['source'],'label':row['label'],'reason':'duplicate_label_conflict_first'})
    else: kept.append(row)
manifest=pd.DataFrame(kept)
if manifest.empty: raise RuntimeError('Tidak ada gambar valid. Periksa struktur ZIP / alias kelas.')
audit=pd.DataFrame(rejected)
manifest.to_csv(REPORT_DIR/'curated_manifest.csv',index=False)
audit.to_csv(REPORT_DIR/'excluded_files.csv',index=False)
print('Data valid:',len(manifest),'| Dikeluarkan:',len(audit))
print(manifest.groupby('label').size().reindex(CLASS_NAMES,fill_value=0).to_string())


## 4 — Preview sampel untuk audit visual (periksa secara manual)

In [ ]:
import matplotlib.pyplot as plt
fig, axes = plt.subplots(len(CLASS_NAMES),4,figsize=(11,14))
for i,label in enumerate(CLASS_NAMES):
    samples=manifest[manifest.label==label].sample(n=min(4,sum(manifest.label==label)),random_state=SEED)
    for j,ax in enumerate(axes[i]):
        ax.axis('off')
        if j < len(samples):
            ax.imshow(Image.open(samples.iloc[j]['path']))
        if j==0: ax.set_title(label,loc='left',fontsize=10)
plt.tight_layout(); plt.savefig(REPORT_DIR/'sample_preview.png',dpi=140); plt.show()
print('Cek hasil preview; jika terlihat kelas salah, perbaiki dataset sebelum melanjutkan.')


## 5 — Split stratified 70% / 15% / 15%
Split ini mencegah kebocoran duplikat **identik**. Jika beberapa foto berasal dari tanaman/sesi kamera yang sama, pisahkan secara **grouped split** dengan metadata asli bila tersedia; tanpa metadata notebook tidak bisa menjamin bebas kebocoran near-duplicate atau per tanaman.

In [ ]:
from sklearn.model_selection import train_test_split
counts=manifest.label.value_counts()
if (counts < 10).any():
    raise RuntimeError('Ada kelas dengan kurang dari 10 gambar setelah audit: '+counts.to_string())
train_df,temp_df=train_test_split(manifest,test_size=0.30,stratify=manifest.label,random_state=SEED)
val_df,test_df=train_test_split(temp_df,test_size=0.50,stratify=temp_df.label,random_state=SEED)
shutil.rmtree(SPLIT_DIR,ignore_errors=True)
for name,df in [('train',train_df),('val',val_df),('test',test_df)]:
    for _,row in df.iterrows():
        target=SPLIT_DIR/name/row.label/Path(row.path).name
        target.parent.mkdir(parents=True,exist_ok=True)
        shutil.copy2(row.path,target)
    df[['label','fingerprint','source']].to_csv(REPORT_DIR/f'{name}_manifest.csv',index=False)
    print(name,len(df),dict(df.label.value_counts()))


## 6 — PyTorch: transfer learning (dua model, split sama)

In [ ]:
import torch, torch.nn as nn
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
import numpy as np, copy
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
import seaborn as sns

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:',device)
# Preprocessing ImageNet pretrained weights (sama untuk kedua arsitektur)
mean=[0.485,0.456,0.406]; std=[0.229,0.224,0.225]
train_tfm=transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE,scale=(0.80,1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ToTensor(),transforms.Normalize(mean,std)])
eval_tfm=transforms.Compose([
    transforms.Resize(256),transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),transforms.Normalize(mean,std)])
train_ds=datasets.ImageFolder(SPLIT_DIR/'train',transform=train_tfm)
val_ds=datasets.ImageFolder(SPLIT_DIR/'val',transform=eval_tfm)
test_ds=datasets.ImageFolder(SPLIT_DIR/'test',transform=eval_tfm)
assert train_ds.classes == val_ds.classes == test_ds.classes
class_names=train_ds.classes
print('Urutan index model:',train_ds.class_to_idx)
train_loader=DataLoader(train_ds,batch_size=BATCH_SIZE,shuffle=True,num_workers=NUM_WORKERS,pin_memory=device.type=='cuda')
val_loader=DataLoader(val_ds,batch_size=BATCH_SIZE,shuffle=False,num_workers=NUM_WORKERS,pin_memory=device.type=='cuda')
test_loader=DataLoader(test_ds,batch_size=BATCH_SIZE,shuffle=False,num_workers=NUM_WORKERS,pin_memory=device.type=='cuda')
with open(MODEL_DIR/'class_names.json','w') as f: json.dump(class_names,f,indent=2)


In [ ]:
from tqdm.auto import tqdm

def make_model(name):
    if name=='mobilenet_v3_large':
        model=models.mobilenet_v3_large(weights=models.MobileNet_V3_Large_Weights.DEFAULT)
        model.classifier[-1]=nn.Linear(model.classifier[-1].in_features,len(class_names))
    elif name=='efficientnet_b0':
        model=models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)
        model.classifier[-1]=nn.Linear(model.classifier[-1].in_features,len(class_names))
    else: raise ValueError(name)
    return model.to(device)

def eval_epoch(model,loader,loss_fn):
    model.eval(); total_loss=0; yt=[]; yp=[]
    with torch.inference_mode():
        for x,y in loader:
            x=x.to(device); y=y.to(device)
            logits=model(x)
            total_loss += loss_fn(logits,y).item()*len(y)
            yt.extend(y.cpu().tolist()); yp.extend(logits.argmax(1).cpu().tolist())
    return total_loss/len(loader.dataset),accuracy_score(yt,yp),f1_score(yt,yp,average='macro',zero_division=0),yt,yp

def train_model(name):
    model=make_model(name)
    optimizer=torch.optim.AdamW(model.parameters(),lr=LEARNING_RATE,weight_decay=1e-4)
    scheduler=torch.optim.lr_scheduler.CosineAnnealingLR(optimizer,T_max=EPOCHS)
    loss_fn=nn.CrossEntropyLoss()
    best_f1=-1; history=[]; best_path=MODEL_DIR/f'{name}_best.pth'
    for epoch in range(1,EPOCHS+1):
        model.train(); running=0
        for x,y in tqdm(train_loader,desc=f'{name} epoch {epoch}/{EPOCHS}',leave=False):
            x=x.to(device); y=y.to(device)
            optimizer.zero_grad(set_to_none=True)
            loss=loss_fn(model(x),y)
            loss.backward(); optimizer.step()
            running+=loss.item()*len(y)
        scheduler.step()
        val_loss,val_acc,val_f1,_,_=eval_epoch(model,val_loader,loss_fn)
        record={'epoch':epoch,'train_loss':running/len(train_ds),'val_loss':val_loss,'val_accuracy':val_acc,'val_macro_f1':val_f1}
        history.append(record)
        print(name,record)
        if val_f1>best_f1:
            best_f1=val_f1
            torch.save({'model_state_dict':model.state_dict(),'classes':class_names,'architecture':name,'img_size':IMG_SIZE,'normalization':{'mean':mean,'std':std}},best_path)
            if DRIVE_ROOT: shutil.copy2(best_path,DRIVE_ROOT/'models'/best_path.name)
    pd.DataFrame(history).to_csv(REPORT_DIR/f'{name}_training_history.csv',index=False)
    return best_path,best_f1

model_names=['efficientnet_b0','mobilenet_v3_large'] if RUN_BOTH_MODELS else ['efficientnet_b0']
results=[]
for name in model_names:
    best_path,best_val_f1=train_model(name)
    results.append({'model':name,'checkpoint':str(best_path),'best_val_macro_f1':best_val_f1})
print(pd.DataFrame(results))


## 7 — Evaluasi test (hanya setelah model final dipilih via validation)

In [ ]:
# Laporkan keduanya sebagai analisis komparatif; jangan pilih arsitektur berdasar skor test.
comparisons=[]
for item in results:
    name=item['model']; path=Path(item['checkpoint'])
    model=make_model(name)
    ckpt=torch.load(path,map_location=device,weights_only=True)
    model.load_state_dict(ckpt['model_state_dict'])
    loss,acc,f1,y_true,y_pred=eval_epoch(model,test_loader,nn.CrossEntropyLoss())
    report=classification_report(y_true,y_pred,target_names=class_names,output_dict=True,zero_division=0)
    pd.DataFrame(report).transpose().to_csv(REPORT_DIR/f'{name}_classification_report.csv')
    cm=confusion_matrix(y_true,y_pred,labels=range(len(class_names)))
    fig,ax=plt.subplots(figsize=(8,6))
    sns.heatmap(cm,annot=True,fmt='d',xticklabels=class_names,yticklabels=class_names,cmap='Blues',ax=ax)
    ax.set(xlabel='Prediksi',ylabel='Aktual',title=name)
    plt.xticks(rotation=35,ha='right'); plt.tight_layout()
    plt.savefig(REPORT_DIR/f'{name}_confusion_matrix.png',dpi=160);plt.show();plt.close(fig)
    comparisons.append({'model':name,'test_accuracy':acc,'test_macro_f1':f1,'best_val_macro_f1':item['best_val_macro_f1']})
summary=pd.DataFrame(comparisons)
summary.to_csv(REPORT_DIR/'model_comparison.csv',index=False)
print(summary.to_string(index=False))
selected=max(results,key=lambda v:v['best_val_macro_f1'])['model']
print('Model terpilih berdasarkan VALIDATION macro F1:',selected)
with open(REPORT_DIR/'selected_model.json','w') as f:
    json.dump({'selected_model':selected,'selection_metric':'best_val_macro_f1','class_names':class_names,'dataset_doi':'10.17632/n8jpkpfz5p.1'},f,indent=2)


## 8 — Simpan seluruh hasil ke Google Drive & contoh prediksi

In [ ]:
if DRIVE_ROOT:
    for f in REPORT_DIR.iterdir():
        if f.is_file(): shutil.copy2(f,DRIVE_ROOT/'results'/f.name)
    for f in MODEL_DIR.iterdir():
        if f.is_file(): shutil.copy2(f,DRIVE_ROOT/'models'/f.name)
    print('Tersimpan di:',DRIVE_ROOT)
else: print('Hasil hanya tersedia di /content/oryza_vision')

# Contoh inferensi satu foto test
winner=selected
model=make_model(winner)
model.load_state_dict(torch.load(MODEL_DIR/f'{winner}_best.pth',map_location=device,weights_only=True)['model_state_dict'])
model.eval()
sample_path=next((SPLIT_DIR/'test').rglob('*.jpg'))
image=Image.open(sample_path).convert('RGB')
with torch.inference_mode():
    probs=torch.softmax(model(eval_tfm(image).unsqueeze(0).to(device)),dim=1)[0].cpu()
idx=int(probs.argmax())
print('Contoh:',sample_path.parent.name,'| Prediksi:',class_names[idx],'| Confidence:',round(float(probs[idx]),3))
print('Confidence bukan kepastian diagnosis; perlu validasi foto lapangan dan uji kalibrasi.')


## Catatan metodologi
- Dataset otomatis berasal dari Hugging Face `Project-AgML/rice_leaf_disease_classification`, yang mengindeks dataset Kaggle `anshulm257/rice-disease-dataset`.
- Dipakai 5 kelas saja. Kelas `Sheath_Blight` dikecualikan dari proyek ini.
- Dataset tidak secara khusus menjamin foto bibit umur 0–30 hari. Uji eksternal foto bibit nyata sangat disarankan.
- Audit otomatis hanya membuang gambar corrupt, gambar kecil, dan duplikat pixel yang identik; duplikat hampir-identik atau perbedaan kondisi tanaman yang sama masih perlu diperiksa.
- Pembagian dilakukan setelah deduplikasi, namun tanpa metadata tanaman/sesi, kebocoran antar tanaman yang difoto berkali-kali tidak dapat dijamin nol.
- Jika training selesai, `reports` dan model dicadangkan ke Google Drive.
